# Results Analysis Notebook

**Project:** DRL for Adaptive Scheduling & Replanning in Dynamic Manufacturing Environments

This notebook walks through:
1. Environment sanity check (random rollout)
2. Baseline comparison
3. Training curve analysis
4. Disruption / replanning analysis
5. Gantt chart visualization
6. Hybrid compute simulation
7. Ablation study template


In [ ]:
import os, sys
# Ensure the project root is on the Python path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Environment Sanity Check

In [ ]:
from src.env.manufacturing_env import ManufacturingEnv

env = ManufacturingEnv()   # default config: 3 nodes, 5 machines each
obs = env.reset(seed=42)

print(f'num_agents   : {env.num_agents}')
print(f'obs_size     : {env.obs_size}')
print(f'action_size  : {env.action_size}')
print(f'Observation shapes: {[o.shape for o in obs]}')

In [ ]:
# Random rollout
total_rewards = []
for step in range(env.max_steps):
    actions = [env.action_spaces[i].sample() for i in range(env.num_agents)]
    obs, rewards, dones, info = env.step(actions)
    total_rewards.append(sum(rewards))
    if all(dones):
        break

print(f'Episode finished at step {step+1}')
print(f'Jobs completed : {info["total_jobs_completed"]}')
print(f'Mean reward/step: {np.mean(total_rewards):.4f}')
print(env.render())

In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(total_rewards, alpha=0.6)
plt.title('Random Policy — Step Rewards')
plt.xlabel('Step')
plt.ylabel('Sum of agent rewards')
plt.grid(linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 2. Baseline Comparison

In [ ]:
from agents.baselines import RandomAgent, FIFOAgent, SPTAgent, EDDAgent, GreedyAgent

def run_episode(agent, env, seed=42):
    obs = env.reset(seed=seed)
    total_reward = 0.0
    done = False
    while not done:
        actions, _, _ = agent.select_actions(obs)
        obs, rewards, dones, info = env.step(actions)
        total_reward += sum(rewards)
        done = all(dones)
    return total_reward, info

baselines = {
    'Random': RandomAgent(env.action_size, env.num_agents),
    'FIFO':   FIFOAgent(env.action_size, env.num_agents, env.M),
    'SPT':    SPTAgent(env.action_size, env.num_agents, env.M, env.K),
    'EDD':    EDDAgent(env.action_size, env.num_agents, env.M, env.K),
    'Greedy': GreedyAgent(env.action_size, env.num_agents, env.M),
}

N_EVAL = 10
baseline_results = {}
for name, agent in baselines.items():
    rewards = [run_episode(agent, env, seed=s)[0] for s in range(N_EVAL)]
    baseline_results[name] = {'mean': np.mean(rewards), 'std': np.std(rewards)}
    print(f'{name:<10}  mean={np.mean(rewards):>8.2f}  std={np.std(rewards):.2f}')

In [ ]:
from visualization.gantt import plot_metrics_comparison

fig = plot_metrics_comparison(
    {n: {'mean_reward': v['mean']} for n, v in baseline_results.items()},
    title='Baseline Comparison — Mean Episode Reward'
)
plt.show()

## 3. Training Curve Analysis

Run training first:
```bash
python -m experiments.train --agent mappo --total-steps 100000
```

Then load results below (or provide synthetic data for demo).

In [ ]:
from visualization.gantt import plot_learning_curves

# --- Synthetic training curves for demo ---
np.random.seed(0)
n_ep = 200
synthetic_rewards = {
    'MAPPO': np.cumsum(np.random.randn(n_ep) * 5 + 0.5).tolist(),
    'GNN':   np.cumsum(np.random.randn(n_ep) * 5 + 0.7).tolist(),
    'FIFO':  (np.ones(n_ep) * -50 + np.random.randn(n_ep) * 2).tolist(),
    'Random':(np.ones(n_ep) * -120 + np.random.randn(n_ep) * 5).tolist(),
}

fig = plot_learning_curves(synthetic_rewards, window=15, title='Training Reward Curves (Demo)')
plt.show()

## 4. Disruption & Replanning Analysis

In [ ]:
from experiments.replan_test import run_replan_episode
from visualization.gantt import plot_disruption_timeline

greedy = GreedyAgent(env.action_size, env.num_agents, env.M)

result = run_replan_episode(
    agent=greedy,
    env=env,
    disruption_step=100,
    failure_fraction=0.5,
    seed=42,
)

print(f"Pre-disruption mean reward : {result['pre_mean_reward']:.2f}")
print(f"Post-disruption mean reward: {result['post_mean_reward']:.2f}")
print(f"Recovery drop              : {result['recovery_drop_pct']:.1f}%")
print(f"Recovery speed             : {result['recovery_speed_steps']} steps")

In [ ]:
fig = plot_disruption_timeline(
    reward_before=result['pre_rewards'],
    reward_after=result['post_rewards'],
    disruption_step=result['disruption_step'],
    title=f"Greedy Agent — Disruption at step {result['disruption_step']}",
)
plt.show()

## 5. Gantt Chart Visualization

In [ ]:
from visualization.gantt import plot_gantt

# Synthetic schedule for demo
np.random.seed(1)
schedule = []
t = 0
for job_id in range(15):
    for op in range(np.random.randint(1, 4)):
        machine_id = np.random.randint(0, 5)
        duration = np.random.uniform(5, 25)
        schedule.append({
            'machine_id': machine_id,
            'job_id': job_id,
            'start': t + np.random.uniform(0, 5),
            'end':   t + np.random.uniform(0, 5) + duration,
            'is_urgent': (job_id % 5 == 0),
        })
    t += np.random.uniform(2, 10)

fig = plot_gantt(schedule, num_machines=5)
plt.show()

## 6. Hybrid Compute Simulation

In [ ]:
from agents.ppo_agent import MAPPOAgent
from hybrid_compute.edge_inference import EdgeInferenceEngine
from hybrid_compute.cloud_trainer import CloudTrainer, FederatedAggregator

# Create a MAPPO agent (untrained, for demo)
agent = MAPPOAgent(obs_size=env.obs_size, action_size=env.action_size, num_agents=env.num_agents)

# Create one edge inference engine per node
edge_engines = [
    EdgeInferenceEngine(
        actor=agent.actor,
        node_id=i,
        base_latency_ms=2.0,
        network_latency_ms=3.0 + i,   # each node has slightly different latency
        simulate_delays=False,          # set True to actually sleep
    )
    for i in range(env.num_agents)
]

# Simulate 5 inference calls per edge node
obs = env.reset(seed=10)
for step in range(5):
    for i, engine in enumerate(edge_engines):
        actions, lp, _ = engine.infer([obs[i]])
        engine.store_experience({'obs': obs[i], 'action': actions[0],
                                  'reward': 0.0, 'next_obs': obs[i],
                                  'done': False, 'log_prob': lp[0]})

# Print edge metrics
for engine in edge_engines:
    m = engine.get_metrics()
    print(f"Node {int(m['node_id'])}: inferences={int(m['n_inferences'])}  "
          f"avg_latency={m['avg_inference_latency_ms']:.1f}ms  "
          f"buffer_size={int(m['exp_buffer_size'])}")

In [ ]:
# Simulate cloud aggregation
cloud = CloudTrainer(
    obs_size=env.obs_size,
    action_size=env.action_size,
    num_agents=env.num_agents,
    bandwidth_mbps=100.0,
)

# Upload experience from each edge node
for engine in edge_engines:
    batch, transfer_ms = engine.upload_experience()
    cloud.receive_experience(node_id=engine.node_id, batch=batch)
    print(f"Node {engine.node_id}: uploaded {len(batch)} transitions, "
          f"transfer={transfer_ms:.2f}ms")

# Cloud training step
metrics = cloud.train_step()
print(f"\nCloud training metrics: {metrics}")

# FedAvg simulation
fed = FederatedAggregator(n_nodes=env.num_agents)
for i in range(env.num_agents):
    fed.submit_local_weights(node_id=i, state_dict=agent.actor.state_dict(), n_samples=50)

agg_weights = fed.aggregate()
print(f"\nFedAvg aggregation complete. Keys: {list(agg_weights.keys())[:3]}...")

## 7. Ablation Study Template

Use this section to record results from ablation experiments.

In [ ]:
from visualization.gantt import plot_ablation

# Replace with your actual ablation results
ablation_results = {
    'MAPPO (full)':          {'mean_reward': 85.2, 'jobs_completed': 42.1},
    'No entropy bonus':      {'mean_reward': 78.4, 'jobs_completed': 39.8},
    'No centralized critic': {'mean_reward': 71.0, 'jobs_completed': 35.2},
    'Reward: no latency':    {'mean_reward': 80.1, 'jobs_completed': 41.0},
    'Reward: no utilization':{'mean_reward': 76.3, 'jobs_completed': 38.5},
    'No disturbances':       {'mean_reward': 92.0, 'jobs_completed': 48.3},
}

fig = plot_ablation(
    ablation_results,
    primary_metric='mean_reward',
    title='Ablation Study — Mean Reward (higher is better)',
)
plt.show()

## 8. Training a MAPPO Agent (Quick Demo)

The cell below runs a **very short** training run (2,000 steps) to verify
the training loop works end-to-end.  For real experiments use
`experiments/train.py` with ≥100k steps.

In [ ]:
from src.env.manufacturing_env import ManufacturingEnv
from agents.ppo_agent import MAPPOAgent

env_demo = ManufacturingEnv()
agent_demo = MAPPOAgent(
    obs_size=env_demo.obs_size,
    action_size=env_demo.action_size,
    num_agents=env_demo.num_agents,
    rollout_steps=256,   # short for demo
    n_epochs=3,
)

obs = env_demo.reset(seed=0)
step_rewards = []

for step in range(512):
    actions, log_probs, values = agent_demo.select_actions(obs)
    global_obs = np.concatenate(obs)
    next_obs, rewards, dones, info = env_demo.step(actions)
    step_rewards.append(sum(rewards))

    agent_demo.store_transition(
        observations=obs,
        global_obs=global_obs,
        actions=actions,
        log_probs=log_probs,
        rewards=rewards,
        dones=dones,
        values=values,
    )
    obs = next_obs

    if agent_demo.buffer.is_ready:
        metrics = agent_demo.update(last_observations=obs, last_dones=dones)
        print(f"  step={step+1:>4}  actor_loss={metrics['actor_loss']:.4f}  "
              f"critic_loss={metrics['critic_loss']:.4f}  "
              f"entropy={metrics['entropy']:.4f}")

    if all(dones):
        obs = env_demo.reset()

print(f"\nDemo training complete. Mean step reward: {np.mean(step_rewards):.3f}")